In [1]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchaudio import transforms, load
from torch.utils.data import DataLoader, Dataset, random_split

In [9]:
import kagglehub

path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'environmental-sound-classification-50' dataset.
Path to dataset files: /kaggle/input/environmental-sound-classification-50


In [10]:
csv_path = '/kaggle/input/environmental-sound-classification-50/esc50.csv'
data_path = '/kaggle/input/environmental-sound-classification-50/audio/audio'

In [11]:
import pandas as pd
df = pd.read_csv(csv_path)
df.head()

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [29]:
classes = sorted(df['category'].unique())
print(classes)

['airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn', 'cat', 'chainsaw', 'chirping_birds', 'church_bells', 'clapping', 'clock_alarm', 'clock_tick', 'coughing', 'cow', 'crackling_fire', 'crickets', 'crow', 'crying_baby', 'dog', 'door_wood_creaks', 'door_wood_knock', 'drinking_sipping', 'engine', 'fireworks', 'footsteps', 'frog', 'glass_breaking', 'hand_saw', 'helicopter', 'hen', 'insects', 'keyboard_typing', 'laughing', 'mouse_click', 'pig', 'pouring_water', 'rain', 'rooster', 'sea_waves', 'sheep', 'siren', 'sneezing', 'snoring', 'thunderstorm', 'toilet_flush', 'train', 'vacuum_cleaner', 'washing_machine', 'water_drops', 'wind']


In [30]:
len(classes)

50

In [14]:
index_to_label = {words: ind for ind, words in enumerate(categories)}

In [15]:
transform = transforms.MelSpectrogram(
    sample_rate = 22050,
    n_mels=32
)
max_len = 500

In [16]:
import torchaudio
class Environmental_sound(Dataset):
    def __init__(self, csv_path, data_path, transforms, max_len):
        self.df = pd.read_csv(csv_path)
        self.data_path = data_path
        self.transforms = transform
        self.max_len = max_len
        self.audios = []

        for file in os.listdir(data_path):
            if file.endswith('.wav'):
                file_path = os.path.join(data_path, file)

                try:
                    category = list(df[df['filename'] == file]['category'])
                    self.audios.append((file_path, category[0]))
                except Exception as e:
                    print(f'Ошибка {e}')

    def __len__(self):
        return len(self.audios)

    def __getitem__(self, ind):
        file_path, category = self.audios[ind]
        waveform, sample_rate = load(file_path)

        if sample_rate != 22050:
            resample = transforms.Resample(orig_freq=sample_rate, new_freq=22050)
            waveform = resample(waveform)

        spectrogram = self.transforms(waveform).squeeze(0)

        if spectrogram.shape[1] > self.max_len:
            spectrogram = spectrogram[:, :self.max_len]

        if spectrogram.shape[1] < self.max_len:
            count_len = self.max_len - spectrogram.shape[1]
            spectrogram = F.pad(spectrogram, (0, count_len))

        return spectrogram, index_to_label[category]

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [18]:
dataset = Environmental_sound(csv_path, data_path, transform, max_len)

train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size

train_data, test_data = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

In [19]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32)

In [20]:
class SimpleCNN(nn.Module):
    def __init__(self,):
        super().__init__()
        self.first = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((8, 8))
        )
        self.second = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 50)
        )
    def forward(self, audio):
        audio = audio.unsqueeze(1)
        audio = self.first(audio)
        audio = self.second(audio)
        return audio

In [21]:
model = SimpleCNN().to(device)

In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [23]:
!pip install -q torchcodec

In [24]:
epochs=180
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Эпоха {epoch+1}/{epochs}, Loss: {running_loss/len(train):.4f}%')

Эпоха 1/180, Loss: 6.7033%
Эпоха 2/180, Loss: 3.9158%
Эпоха 3/180, Loss: 3.9139%
Эпоха 4/180, Loss: 3.9143%
Эпоха 5/180, Loss: 3.9134%
Эпоха 6/180, Loss: 3.9131%
Эпоха 7/180, Loss: 3.9134%
Эпоха 8/180, Loss: 3.9134%
Эпоха 9/180, Loss: 3.9132%
Эпоха 10/180, Loss: 3.9135%
Эпоха 11/180, Loss: 3.9141%
Эпоха 12/180, Loss: 3.9138%
Эпоха 13/180, Loss: 3.9135%
Эпоха 14/180, Loss: 3.9133%
Эпоха 15/180, Loss: 3.9137%
Эпоха 16/180, Loss: 3.9130%
Эпоха 17/180, Loss: 3.9135%
Эпоха 18/180, Loss: 3.9137%
Эпоха 19/180, Loss: 3.9132%
Эпоха 20/180, Loss: 3.9135%
Эпоха 21/180, Loss: 3.9138%
Эпоха 22/180, Loss: 3.9135%
Эпоха 23/180, Loss: 3.9137%
Эпоха 24/180, Loss: 3.9133%
Эпоха 25/180, Loss: 3.9135%
Эпоха 26/180, Loss: 3.9136%
Эпоха 27/180, Loss: 3.9136%
Эпоха 28/180, Loss: 3.9137%
Эпоха 29/180, Loss: 3.9138%
Эпоха 30/180, Loss: 3.9134%
Эпоха 31/180, Loss: 3.9135%
Эпоха 32/180, Loss: 3.9135%
Эпоха 33/180, Loss: 3.9137%
Эпоха 34/180, Loss: 3.9128%
Эпоха 35/180, Loss: 3.9134%
Эпоха 36/180, Loss: 3.9126%
Э

In [25]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy на тесте: {100 * correct / total:.2f}%')

Accuracy на тесте: 1.00%


In [32]:
from google.colab import files

torch.save(classes, 'labels_environmental_sound_classification_50.pth')
torch.save(model.state_dict(), 'model_environmental_sound_classification_50.pth')

files.download('labels_environmental_sound_classification_50.pth')
files.download('model_environmental_sound_classification_50.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>